# Panel de Longevidad — Interpretación de Biomarcadores
**Laboratorio de Análisis Clínicos · Área de Longevidad · Medicina 3.0**

Ejecutá la celda de abajo con **Run Cell** (▷) para abrir la interfaz.

In [ ]:
import sys, os, csv
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Path setup (Positron / VS Code / Jupyter)
_nb_file = globals().get("__vsc_ipynb_file__") or globals().get("__file__", None)
nb_dir = Path(_nb_file).parent.resolve() if _nb_file else Path(os.getcwd()).resolve()
if str(nb_dir) not in sys.path:
    sys.path.insert(0, str(nb_dir))

# Cargar .env si existe
_env = nb_dir / ".env"
if _env.exists():
    for _ln in _env.read_text().splitlines():
        if "=" in _ln and not _ln.strip().startswith("#"):
            _k, _v = _ln.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())

# ──────────────────────────────────────────────────────────
#  WIDGETS
# ──────────────────────────────────────────────────────────

w_header = widgets.HTML("""
<div style="background:linear-gradient(135deg,#085041 0%,#0F6E56 100%);
            padding:18px 24px;border-radius:10px;margin-bottom:4px">
  <div style="display:flex;align-items:center;gap:14px">
    <span style="font-size:32px">🧬</span>
    <div>
      <div style="color:#fff;font-size:18px;font-weight:700;font-family:sans-serif;letter-spacing:.3px">
        Panel de Longevidad
      </div>
      <div style="color:#5DCAA5;font-size:12px;font-family:sans-serif;margin-top:3px">
        Laboratorio · Área de Longevidad · Medicina 3.0
      </div>
    </div>
  </div>
</div>
""")

_S = {"description_width": "130px"}
_W = widgets.Layout

# CSV
w_csv = widgets.Text(
    value="muestra_panel.csv",
    description="CSV del LIS:",
    placeholder="ruta/al/archivo.csv",
    layout=_W(width="440px"),
    style=_S,
)

# Modo
w_modo = widgets.ToggleButtons(
    options=[("  Scoring  ", "scoring"), ("  LLM + Claude  ", "llm")],
    value="scoring",
    description="Modo:",
    tooltips=["Clasificación determinística — sin API",
              "Scoring + narrativa clínica integrada vía Claude"],
    style={"description_width": "60px", "button_width": "148px"},
)

# Modelo
w_model = widgets.Dropdown(
    options=[
        ("Sonnet 4.6 — recomendado",         "claude-sonnet-4-6"),
        ("Haiku 4.5 — rápido / económico",   "claude-haiku-4-5-20251001"),
        ("Opus 4.7 — máxima calidad",        "claude-opus-4-7"),
    ],
    value="claude-sonnet-4-6",
    description="Modelo Claude:",
    layout=_W(width="400px"),
    style=_S,
)

# API key
w_apikey = widgets.Password(
    value=os.environ.get("ANTHROPIC_API_KEY", ""),
    placeholder="sk-ant-...  (o configurar en .env)",
    description="API Key:",
    layout=_W(width="400px"),
    style=_S,
)

_llm_label = widgets.HTML(
    "<div style='font-size:11px;color:#085041;font-weight:600;"
    "margin-bottom:6px;text-transform:uppercase;letter-spacing:.5px'>Opciones LLM</div>"
)
w_llm_box = widgets.VBox(
    [_llm_label, w_model, w_apikey],
    layout=_W(border="1px solid #5DCAA5", padding="12px 16px",
              border_radius="8px", margin="6px 0", display="none"),
)

def _on_modo(change):
    w_llm_box.layout.display = "" if change["new"] == "llm" else "none"

w_modo.observe(_on_modo, names="value")

# Paciente + carpeta
w_patient = widgets.Text(
    value="",
    placeholder="Dejar vacío = todos",
    description="Paciente ID:",
    layout=_W(width="310px"),
    style=_S,
)
w_outdir = widgets.Text(
    value="./informes",
    description="Carpeta salida:",
    layout=_W(width="280px"),
    style=_S,
)
w_pdf = widgets.Checkbox(value=True, description="Generar PDFs", indent=False,
                         layout=_W(margin="4px 0"))

# Botón
w_btn = widgets.Button(
    description="Procesar panel",
    button_style="success",
    icon="play",
    layout=_W(width="180px", height="38px", margin="10px 0 2px"),
)
w_status = widgets.HTML("")

# Output
w_out = widgets.Output(
    layout=_W(border="1px solid #D3D1C7", padding="16px",
              border_radius="8px", margin_top="10px", min_height="80px")
)

# ──────────────────────────────────────────────────────────
#  LÓGICA DE PROCESAMIENTO
# ──────────────────────────────────────────────────────────

_COLORES = {
    "Óptimo":   "background-color:#EAF3DE;color:#27500A",
    "Bueno":    "background-color:#E1F5EE;color:#085041",
    "Atención": "background-color:#FAEEDA;color:#854F0B",
    "Riesgo":   "background-color:#FAECE7;color:#993C1D",
    "Crítico":  "background-color:#FCEBEB;color:#A32D2D;font-weight:bold",
}

def _color_nivel(v):
    return _COLORES.get(str(v), "")

def _run_pipeline():
    import pandas as pd
    from datetime import datetime
    from scoring_engine import procesar_paciente, SCORE_MAP
    from report_generator import generate_pdf
    from main import call_llm

    modo    = w_modo.value
    model   = w_model.value
    api_key = w_apikey.value.strip()
    pid_fil = w_patient.value.strip() or None
    out_dir = Path(w_outdir.value)

    csv_raw = w_csv.value.strip()
    csv_path = Path(csv_raw) if Path(csv_raw).is_absolute() else nb_dir / csv_raw

    if not csv_path.exists():
        print(f"[ERROR] Archivo no encontrado:\n  {csv_path}")
        return

    if api_key:
        os.environ["ANTHROPIC_API_KEY"] = api_key

    if modo == "llm" and not os.environ.get("ANTHROPIC_API_KEY"):
        print("[ERROR] Modo LLM requiere API Key.\n"
              "Ingresala en el campo 'API Key' o creá un archivo .env con ANTHROPIC_API_KEY=sk-ant-...")
        return

    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))

    if pid_fil:
        rows = [r for r in rows if r.get("paciente_id") == pid_fil]
        if not rows:
            print(f"[ERROR] Paciente '{pid_fil}' no encontrado en el CSV.")
            return

    model_label = f" | {model}" if modo == "llm" else ""
    print(f"Pacientes: {len(rows)} | Modo: {modo.upper()}{model_label}")
    print("═" * 60)

    out_dir.mkdir(parents=True, exist_ok=True)
    rows_map = {r["paciente_id"]: r for r in rows}
    pdfs = []

    for row in rows:
        panel = procesar_paciente(row)
        pid   = panel.paciente_id

        # ── Cabecera del paciente
        alerta_str = f"  ⚠ {len(panel.alertas)} alerta(s)" if panel.alertas else ""
        print(f"\n▸ Paciente: {pid}  |  {panel.edad} años  |  Sexo: {panel.sexo}  |  {panel.fecha}")
        print(f"  Score global: {panel.score_global} ({panel.score_global_numerico:.2f}){alerta_str}")
        print(f"  Edad biológica estimada: {panel.edad_biologica_estimada}")

        # ── Tabla de categorías
        filas = []
        for cat in panel.categorias:
            peor = max(
                (b for b in cat.biomarcadores if b.clasificacion and b.valor is not None),
                key=lambda b: SCORE_MAP.get(b.clasificacion, 0), default=None
            )
            delta_str = ""
            if peor and peor.delta is not None:
                signo = "+" if peor.delta > 0 else ""
                delta_str = f" ({signo}{peor.delta} {peor.tendencia})"
            filas.append({
                "Categoría":   cat.nombre_display,
                "Score":       cat.score,
                "N":           len([b for b in cat.biomarcadores if b.valor is not None]),
                "Hallazgo clave": (f"{peor.nombre}: {peor.valor} {peor.unidad}{delta_str}"
                                   if peor else "—"),
                "⚠": "⚠" if any(b.es_alerta for b in cat.biomarcadores) else "",
            })
        df_cats = pd.DataFrame(filas)
        display(df_cats.style
                .applymap(_color_nivel, subset=["Score"])
                .hide(axis="index"))

        # ── Patrones
        if panel.patrones:
            print("\nPatrones inter-sistema:")
            for p in panel.patrones:
                ic = "🔴" if p["relevancia"] == "Alta" else "🟡"
                print(f"  {ic} {p['patron']}")
                print(f"     → {p['descripcion']}")

        # ── Alertas
        if panel.alertas:
            print("\nAlertas críticas:")
            for a in panel.alertas:
                print(f"  ⚠  {a.nombre}: {a.valor} {a.unidad}")
                print(f"     → {a.nota_alerta}")

        # ── LLM
        llm_res = None
        if modo == "llm":
            print(f"\n  Consultando Claude ({model})...")
            try:
                llm_res = call_llm(panel, rows_map[pid], model=model)
                resumen = llm_res.get("resumen_ejecutivo", "—")
                narrativa = llm_res.get("narrativa_clinica", "")
                print(f"\n  RESUMEN EJECUTIVO:\n  {resumen}")
                if narrativa:
                    print(f"\n  NARRATIVA CLÍNICA:\n  {narrativa}")
                prios = llm_res.get("protocolo_dire", {}).get("prioridades", [])
                if prios:
                    print("\n  PROTOCOLO DIRe:")
                    for pr in prios:
                        print(f"    {pr['orden']}. {pr['objetivo']}")
                        for iv in pr.get("intervenciones", [])[:3]:
                            nota = f" [{iv['nota_arg']}]" if iv.get("nota_arg") else ""
                            print(f"       • [{iv['tipo']}] {iv['descripcion']}{nota}")
                plan = llm_res.get("plan_reevaluacion", {})
                if plan.get("plazo_sugerido"):
                    print(f"\n  Reevaluación sugerida: {plan['plazo_sugerido']}")
            except Exception as e:
                print(f"  [ERROR LLM] {e}")

        # ── PDF
        if w_pdf.value:
            from datetime import datetime
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            out_path = str(out_dir / f"informe_{pid}_{modo}_{ts}.pdf")
            generate_pdf(panel, out_path, modo, llm_res)
            pdfs.append(out_path)
            print(f"\n  PDF → {out_path}")

        print("\n" + "─" * 60)

    if pdfs:
        print(f"\n✓ {len(pdfs)} informe(s) generado(s) en:\n  {out_dir.resolve()}")
    else:
        print("\n✓ Procesamiento completado (PDFs desactivados).")


def _on_click(_):
    w_status.value = "<span style='color:#085041;font-size:13px'>⏳ Procesando...</span>"
    w_btn.disabled = True
    w_out.clear_output(wait=True)
    with w_out:
        try:
            _run_pipeline()
        except Exception as ex:
            import traceback
            print(f"[ERROR inesperado] {ex}")
            traceback.print_exc()
        finally:
            w_btn.disabled = False
            w_status.value = ""

w_btn.on_click(_on_click)

# ──────────────────────────────────────────────────────────
#  LAYOUT
# ──────────────────────────────────────────────────────────
display(widgets.VBox([
    w_header,
    widgets.VBox([
        w_csv,
        widgets.HBox(
            [widgets.Label("Modo:", layout=_W(width="64px", margin="5px 0")), w_modo]
        ),
        w_llm_box,
        widgets.HBox([w_patient, widgets.Label("  "), w_outdir]),
        w_pdf,
    ], layout=_W(padding="12px 4px")),
    widgets.HBox([w_btn, widgets.Label("  "), w_status],
                 layout=_W(align_items="center")),
    w_out,
], layout=_W(max_width="700px")))